In [ ]:
def gcd_extended(a, b):
    """拡張ユークリッドの互除法: ax + by = gcd(a, b) となる (g, x, y) を返す"""
    # gはgcd(a,b)
    if a == 0:
        return b, 0, 1
    g, x1, y1 = gcd_extended(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return g, x, y

def solve_linear_congruence_scalar(a, b, m):
    """単一の 1次合同式 ax ≡ b (mod m) を解く"""
    g, x, y = gcd_extended(a, m)
    if b % g != 0:
        return None  # 解なし
    # 特殊解を一つ求める
    return (x * (b // g)) % (m // g)

class SmithNormalForm:
    def __init__(self, A):
        self.A = [row[:] for row in A]  # 行列のコピー
        self.rows = len(A)
        self.cols = len(A[0])
        # 変換行列 U, V (Identity matrices)
        self.U = [[1 if i == j else 0 for j in range(self.rows)] for i in range(self.rows)]
        self.V = [[1 if i == j else 0 for j in range(self.cols)] for i in range(self.cols)]

    def _swap_rows(self, i, j):
        self.A[i], self.A[j] = self.A[j], self.A[i]
        self.U[i], self.U[j] = self.U[j], self.U[i]

    def _swap_cols(self, i, j):
        for r in range(self.rows):
            self.A[r][i], self.A[r][j] = self.A[r][j], self.A[r][i]
        for r in range(self.cols):
            self.V[i][r], self.V[j][r] = self.V[j][r], self.V[i][r]

    def _add_row(self, src, dst, k):
        """dst = dst + k * src"""
        for c in range(self.cols):
            self.A[dst][c] += k * self.A[src][c]
        for c in range(self.rows):
            self.U[dst][c] += k * self.U[src][c]

    def _add_col(self, src, dst, k):
        """dst = dst + k * src"""
        for r in range(self.rows):
            self.A[r][dst] += k * self.A[r][src]
        for r in range(self.cols):
            self.V[src][r] += k * self.V[dst][r] # Vの更新は逆行列の性質上注意

    def compute(self):
        """スミス標準形 S = U * A * V を計算する"""
        for k in range(min(self.rows, self.cols)):
            while True:
                # 1. (k,k)成分を最小の非ゼロにする
                pivot_found = False
                min_val = float('inf')
                pivot_pos = (k, k)
                
                for r in range(k, self.rows):
                    for c in range(k, self.cols):
                        if self.A[r][c] != 0 and abs(self.A[r][c]) < min_val:
                            min_val = abs(self.A[r][c])
                            pivot_pos = (r, c)
                            pivot_found = True
                
                if not pivot_found: break
                
                self._swap_rows(k, pivot_pos[0])
                self._swap_cols(k, pivot_pos[1])

                # 2. 第k列の掃き出し（ユークリッドの互除法的ステップ）
                changed = False
                for r in range(k + 1, self.rows):
                    if self.A[r][k] != 0:
                        q = self.A[r][k] // self.A[k][k]
                        self._add_row(k, r, -q)
                        if self.A[r][k] != 0: changed = True
                
                # 3. 第k行の掃き出し
                for c in range(k + 1, self.cols):
                    if self.A[k][c] != 0:
                        q = self.A[k][c] // self.A[k][k]
                        self._add_col(k, c, -q)
                        if self.A[k][c] != 0: changed = True
                
                if not changed: break

        return self.A, self.U, self.V

def solve_system_mod(A, b, P):
    """Ax ≡ b (mod P) を解く"""
    snf = SmithNormalForm(A)
    S, U, V = snf.compute()
    
    # U * A * V = S  =>  A = U^-1 * S * V^-1
    # Ax = b  =>  U^-1 * S * V^-1 * x = b
    # S * (V^-1 * x) = U * b (mod P)
    
    # Ub を計算
    Ub = [0] * len(U)
    for i in range(len(U)):
        for j in range(len(b)):
            Ub[i] = (Ub[i] + U[i][j] * b[j]) % P
            
    # S * z = Ub (mod P) を解く
    z = [0] * len(S[0])
    for i in range(min(len(S), len(S[0]))):
        val = solve_linear_congruence_scalar(S[i][i], Ub[i], P)
        if val is None:
            return None # 解なし
        z[i] = val
        
    # x = V * z (mod P) を計算
    x = [0] * len(V)
    for i in range(len(V)):
        for j in range(len(z)):
            x[i] = (x[i] + V[i][j] * z[j]) % P
    return x

# 使用例
A = [[2, 1], [1, 2]]
b = [1, 1]
P = 768
solution = solve_system_mod(A, b, P)
print(f"解 x: {solution}")